In [1]:
pdf_loc = r"C:\Users\rajne\Desktop\14.pdf"

import re
import pdfplumber
import pandas as pd

# Group 1: 1 or 2 digits for hours | Group 2: dot or colon | Group 3: exactly 2 digits for minutes
time_pattern = re.compile(r'(\d{1,2})[.:](\d{2})')

def normalize_time_cell(raw_string):
    # Null checking 
    if not raw_string or str(raw_string).strip() in ('', 'None', '...', None):
        return None
        
    clean_time_string = str(raw_string).strip().replace('\n', ' ')

    match = time_pattern.findall(clean_time_string)
    
    if not match:
        return None  # Safely filters out junk characters, letters, or '...' leaks
        
    formatted_times = [f"{int(hour):02d}:{minute}" for hour, minute in match]
    
    # THE SEQUENTIAL TIMELINE MAPPER (Departure-Centric cuz arrival time is incositent in tables)
    # If the cell block yielded exactly 2 time markers (e.g., ['06:00', '07:35']):
    if len(formatted_times) == 2:
        return {
            'A': formatted_times[0], # First chronologically is Arrival time 
            'D': formatted_times[1]  # Second chronologically is Departure time
        }
    
    # If the cell block consist exactly 1 time marker (e.g., ['07:40']):
    elif len(formatted_times) == 1:
        # Check if the block is an outbound station line or final terminus
        # Since we are building a departure-centric routing model, 
        # map the isolated single element as Departure ('D') by default!
        return {'D': formatted_times[0]}
        
    return None
# ---------------------------------------


with pdfplumber.open(pdf_loc) as pdf:
    pages = pdf.pages

    trains_details = []

    for page in pages:
        # Extract graphical vertical grid lines using pdfplumber inbuilt function ".lines"
        grid_lines = page.lines

        # Extracting explicit X positions each of the vertical grid lines, this will act as an absolute boundary ANCHOR, TO HANDLE TEXT BLEED/OVERFLOW.
        x_positions = []
        y_positions = []
        line_top_values = []
        line_bottom_values = []
        
        for line in grid_lines:
            # A vertical line is longer than it is width
            if line["height"] > line["width"] and line["height"] > 20:
                x_positions.append(line["x0"])  

            if line["width"] > line["height"]:
                y_positions.append(line["y0"])

            line_top_values.append(line["top"])
            line_bottom_values.append(line["bottom"])
 
        x_positions = sorted(list(set(x_positions)))    # De-duplicate and sort coordinates left-to-right & top-to-bottom
        y_positions = sorted(list(set(y_positions)))

        left_border = x_positions[0]
        right_border = x_positions[-1]
        top_border = min(line_top_values)
        bottom_border = max(line_bottom_values)

        # Applying crop setting to page, to isolate table portion
        bbox = (left_border - 5, top_border - 5, right_border + 9, bottom_border + 9)
        table_area_page = page.crop(bbox)

        
        # Configure the extraction OF TABLE to follow the extracted grid data above, not the text flow! ()
        custom_settings = {
            "vertical_strategy": "explicit",# by default this setting is text, which works on the basis of text and gap/space btwn them to recognize
                                            # cell change
            "explicit_vertical_lines": x_positions,
            "horizontal_strategy": "lines",  
            "snap_tolerance": 2, # USEFULL: In case PDFs are poorly aligned(text/data bleed) > If a piece of text is within 3(given numeric value) pixels
                                            # of a grid line, snap it backward inside the border where it belongs.
            "join_tolerance": 3,  # To manage grid line style(if any): If a line is broken into tiny fragments (dash-dash-dash) instead of being a solid
                                  # line, this setting bridges gaps up to 3 pixels wide, turning them into one continuous solid line boundary.
        }
        
        # 4. Extract data cleanly into structured lists
        table_data = table_area_page.extract_table(table_settings=custom_settings)
        
#-----------------------------------------------------------------
        trn_name_row_position = None    # trn_name_row_position = 1; is default/mostly right
        trn_number_row_position = None  # trn_name_row_position + 1   # trn_number_row_position = 2; is default
        start_data_row = None           # trn_name_row_position + 4
#-----------------------------------------------------------------      
        
        if table_data:
            rows_count = len(table_data)
            columns_count = len(table_data[0])

            data_row_index = 0
            col_start= 0
            for row_index, row_data in enumerate(table_data[:7]):
                position_zero = False
                if 'TRAIN NAME' in row_data or 'RAIN NAME' in row_data:   
                    data_row_index = row_index
                   
                    if row_index < 2:
                        if row_data.index('TRAIN NAME') == 0:
                            col_start = 1
                            position_zero = True
                        else:
                            col_start = 2
                    else:
                        col_start = 3
                    break

            print(page)
            for col in range(col_start, columns_count):
                
                train_name = str(table_data[data_row_index][col]).strip().replace('\n',' ')
                
                train_number = str(table_data[data_row_index + 1][col]).replace('*','').replace('**','').strip()
                # print(train_name, train_number)

                
                data_start_row = data_row_index + 5
                
                for row in table_data[data_start_row : rows_count - 2]:
                    if len(row) == columns_count:

                        if position_zero:
                            omit = ' 0123456789'
                            raw_station_name = str(row[0]).replace('\n', ' ').replace('Km.','').strip(omit) #.split(' ', 1)
                           
                            # if raw_station_name[0].isdigit():
                            #     stn_name = raw_station_name[1]
                            # else:
                            stn_name = raw_station_name
                            station_name = stn_name.replace('\n', ' ').strip()
                            # print(station_name)
                            
                        else:
                            raw_station_name = row[1]
                            station_name = str(raw_station_name).replace('\n', ' ').strip()

                        raw_time_cell_value = row[col]
                        processed_dtime = normalize_time_cell(raw_time_cell_value)
                        
                        if processed_dtime:
                            dtime = processed_dtime.get('D') 
                            trains_details.append([train_number, train_name, station_name, dtime])
                        else:
                            continue


schedules_test_df = pd.DataFrame(trains_details, columns=['train_num', 'train_name', 'halt_station_name', 'departure_time'])

to_file = r"D:\01_Research_and_Development\0_Python_Grind\Mini Projects\trains_schedule_pdf_14.csv"

schedules_test_df.to_csv(to_file, index = False)
# for details in trains_details:
#     print(details)
print('File created!')

<Page:1>
<Page:2>
<Page:3>
<Page:4>


ValueError: Bounding box (35.1851, 36.33439999999996, 891.9917, 696.2802) is not fully within parent page bounding box (0.0, 0.0, 583.937, 728.504)

---
---
---

# MORE RECTIFIED SCRIPT:

In [1]:
pdf_loc = r"C:\Users\rajne\Desktop\pdf3.pdf"

import re
import pdfplumber
import pandas as pd

def extract_table(pdf_loc):
    # Group 1: 1 or 2 digits for hours | Group 2: dot or colon | Group 3: exactly 2 digits for minutes
    time_pattern = re.compile(r'(\d{1,2})[.:](\d{2})')
    
    def normalize_time_cell(raw_string):
        # Null checking 
        if not raw_string or str(raw_string).strip() in ('', 'None', '...', None):
            return None
            
        clean_time_string = str(raw_string).strip().replace('\n', ' ')
    
        match = time_pattern.findall(clean_time_string)
        
        if not match:
            return None  # Safely filters out junk characters, letters, or '...' leaks
            
        formatted_times = [f"{int(hour):02d}:{minute}" for hour, minute in match]
        
        # THE SEQUENTIAL TIMELINE MAPPER (Departure-Centric cuz arrival time is incositent in tables)
        # If the cell block yielded exactly 2 time markers (e.g., ['06:00', '07:35']):
        if len(formatted_times) == 2:
            return {
                'A': formatted_times[0], # First chronologically is Arrival time 
                'D': formatted_times[1]  # Second chronologically is Departure time
            }
        
        # If the cell block consist exactly 1 time marker (e.g., ['07:40']):
        elif len(formatted_times) == 1:
            # Check if the block is an outbound station line or final terminus
            # Since we are building a departure-centric routing model, 
            # map the isolated single element as Departure ('D') by default!
            return {'D': formatted_times[0]}
            
        return None
    # ---------------------------------------
    
    with pdfplumber.open(pdf_loc) as pdf:
        pages = pdf.pages
    
        trains_details = []
    
        for page in pages:
        # Extract graphical vertical grid lines using pdfplumber inbuilt function ".lines"
            grid_lines = page.lines
        
            # Extracting explicit X positions each of the vertical grid lines, this will act as an absolute boundary ANCHOR, TO HANDLE
            # TEXT BLEED/OVERFLOW.
            x_positions = []
            y_positions = []
            line_top_values = []
            line_bottom_values = []
            
            for line in grid_lines:
                # A vertical line is longer than it is width
                if line["height"] > line["width"] and line["height"] > 20:
                    x_positions.append(line["x0"])  
        
                if line["width"] > line["height"]:
                    y_positions.append(line["y0"])
        
                line_top_values.append(line["top"])
                line_bottom_values.append(line["bottom"])
        
            x_positions = sorted(list(set(x_positions)))    # De-duplicate and sort coordinates left-to-right & top-to-bottom
            y_positions = sorted(list(set(y_positions)))
        
            left_border = x_positions[0]
            right_border = x_positions[-1]
            top_border = min(line_top_values)
            bottom_border = max(line_bottom_values)
    
            # TO HANDLE BOUNDARY ERROR:
            safe_x0 = max(0, left_border - 5)
            safe_top = max(0, top_border - 5)
            safe_x1 = min(page.width, right_border + 9)
            safe_bottom = min(page.height, bottom_border + 9)
    
            # Applying the safe, guarded crop setting to isolate the table portion safely
            bbox = (safe_x0, safe_top, safe_x1, safe_bottom)
            table_area_page = page.crop(bbox)
        
            
            # Configure the extraction OF TABLE to follow the extracted grid data above, not the text flow! ()
            custom_settings = {
                "vertical_strategy": "explicit", # by default this setting is text, which works on the basis of text and gap/space btwn them to
                                                 # recognize cell change
                "explicit_vertical_lines": x_positions,
                "horizontal_strategy": "lines",  
                "snap_tolerance": 2, # USEFULL: In case PDFs are poorly aligned(text/data bleed) > If a piece of text is within 3(given numeric value)
                                                # pixels of a grid line, snap it backward inside the border where it belongs.
                "join_tolerance": 3,  # To manage grid line style(if any): If a line is broken into tiny fragments (dash-dash-dash) instead of 
                        # being a solid line, this setting bridges gaps up to 3 pixels wide, turning them into one continuous solid line boundary.
            }
            
            raw_table_data = table_area_page.extract_table(table_settings = custom_settings)
        # --------------------------------------------------------------------------------------------------------------------------
        
            #figuring to merge and skipp none values rows:
            cleaned_table_rows = []     # list to store clean extracted data rows
            
            total_table_rows = len(raw_table_data)
            row_cursor = 0
             
            while row_cursor < total_table_rows:
                data_row = raw_table_data[row_cursor]
                
                # null_value_count = 0
                # for val in data_row:
                #     if val in ['', None]:
                #         null_value_count += 1 
                # if null_value_count == len(raw_table_data[0]):
                #     row_cursor += 1
                #     continue
                
                current_row_index = row_cursor
                next_row_index = current_row_index + 1
        
                merged_row = []
                
                while next_row_index < total_table_rows:
                    next_row_data = raw_table_data[next_row_index]
                    
                    if not merged_row:
                        on_data = data_row
                    else: 
                        on_data = merged_row
                        
                    if not on_data:
                        next_row_index += 1    # updating next_row_index cuz later on basis of this will move the main row cursor to new data line in
                        continue               # master loop 
                        
                    first_three_items = next_row_data[:3]
                    is_empty_row = False
                    empty_count = 0
                    
                    for item in first_three_items:
                        if item in [None,'']:
                            empty_count += 1 
                        if empty_count == 3:
                            is_empty_row = True
                            
                    if is_empty_row:
                        # temp_row = []
                        for i in range(len(raw_table_data[0])):
                            if on_data[i]:
                                merged_row.append(on_data[i])
                                
                            elif next_row_data[i]:
                                merged_row.append(next_row_data[i])
        
                            else:
                                merged_row.append('')
                        next_row_index += 1   # Incrementing to check for more next rows
                    else:
                        break  # No empty row found then, stop and move to new data row
                
                if merged_row:
                    cleaned_table_rows.append(merged_row)
                else:
                    cleaned_table_rows.append(data_row)  
                
                row_cursor = next_row_index   # updating data row's index value to next row in master/table loop
                                   
     # --------------------------------------------------------------------------------------------------------------------------        
            if cleaned_table_rows:
                # print(cleaned_table_rows)
                rows_count = len(cleaned_table_rows)
                columns_count = len(cleaned_table_rows[0])
        
                data_row_index = 0
                col_start = 2 # default value that works for most pages
                position_zero = False
                
                for row_index, row_data in enumerate(cleaned_table_rows[:8]):
                    
                    # list is causing error and is very inconsistent so Flattening list to single uppercase block text string
                    # 1. Start with an empty list to collect individual cell texts
                    row_text_ls = []
                    for row_text in row_data:
                        if row_text:  # Check if the cell has content (skips None and empty strings) 
                            row_text = str(row_text).upper()
                            row_text_ls.append(row_text) # Convert to string and add it to our collection list
                    
                    row_as_one_string = ' '.join(row_text_ls) # Joining list of words together ONCE outside the loop
    
                    if 'TRAIN NAME' in row_as_one_string or 'RAIN NAME' in row_as_one_string:   
                        data_row_index = row_index
                       
                        if row_index < 2:
                            # Match index securely out of flat string checks
                            if row_data[0]:
                                first_part_str = str(row_data[0]).upper()
                            else:
                                first_part_str = ""
                            
                            if 'TRAIN' in first_part_str or 'NAME' in first_part_str:
                                col_start = 1
                                position_zero = True
                            else:
                                col_start = 2
                        else:
                            col_start = 3
                        break
            
                print(page)
                if data_row_index + 1 >= rows_count:  # this is boundary check while processing last row of the table, to avoid index error crash
                    continue
    
                for col in range(col_start, columns_count):     
                    # To check If the current column index exceeds the actual length of either the Train Name or Train Number row, 
                    # skip it to handle any IndexError. This filters out misaligned text or leaking columns.        
                    if col >= len(cleaned_table_rows[data_row_index]) or col >= len(cleaned_table_rows[data_row_index + 1]):
                        continue 
                                
                    train_name = str(cleaned_table_rows[data_row_index][col]).strip().replace('\n',' ')
                    raw_train_number = str(cleaned_table_rows[data_row_index + 1][col])
                    train_number = re.sub(r'[\*#]+', '', raw_train_number)
                    # print(train_name, train_number)
               
                    data_start_row = data_row_index + 5
        
                    for row in cleaned_table_rows[data_start_row : rows_count - 2]:
                        first_three_cell_items = [str(merging_cell)for merging_cell in row[:3]]
                    
                        first_three_cell_string = ' '.join(first_three_cell_items)
                        stn_name = re.sub(r'^(\d+\s*|\bKm\.?\s+|\b[ad]\b\s+)+', '', first_three_cell_string, flags=re.IGNORECASE)
                        stn_name = re.split(r'\s+(?=\d|\.\d)', stn_name, maxsplit = 1)[0].strip()
                        stn_name = re.sub(r'\s+\b(a\s+d|a/d|a|d|ad|arr\.?|dep\.?)\b\s*$', '', stn_name) # TO REMOVE BLEEDING TIME CELL PREFIXES
                        stn_name = re.sub(r'\s*\/\s*$', '', stn_name)
                        station_name = re.sub(r'\s*[\.\s]{2,}\s*', ' ', stn_name).replace('\n', ' ')   # TO REMOVE TRAILING DOTS 
 
                        raw_time_cell_value = row[col]
                        processed_dtime = normalize_time_cell(raw_time_cell_value)
                        
                        if processed_dtime:
                            dtime = processed_dtime.get('D') 
                            trains_details.append([train_number, train_name, station_name, dtime])
                        else:
                            continue
    
        
    schedules_test_df = pd.DataFrame(trains_details, columns=['train_num', 'train_name', 'halt_station_name', 'departure_time'])
        
    return schedules_test_df
    
    for details in trains_details:
        print(details)
        
    # print('File created!')

In [2]:
from_file = r"C:\Users\rajne\Desktop\18.pdf"

to_file = r"D:\01_Research_and_Development\0_Python_Grind\Datasets\trains_schedule_pdf_18.csv"
got_data_df =  extract_table(from_file)
got_data_df.to_csv(to_file, index = False)


# for details in trains_details:
#     print(details)       
print('File created!')

<Page:1>
<Page:2>
<Page:3>
<Page:4>
<Page:5>
<Page:6>
<Page:7>
<Page:8>
File created!


---
---

In [63]:
import pdfplumber
import pandas as pd
stn_code_pdf = r"C:\Users\rajne\Desktop\Station_Code_Index_1.pdf"
with pdfplumber.open(stn_code_pdf) as stn_codef:
    page = stn_codef.pages[3]

    station_name_codes_master = {}
    station_name_codes_master['station_name'] = []
    station_name_codes_master['station_code'] = []

    standby_ls = []
    # for page in pages:
    
    table = page.extract_table()
    standard_len = len(table[1])
    for row in table[1::2]:
        row_len = len(row)
        print(row)
        if row_len == standard_len:
            for index, pair in enumerate(row):
                if pair == None:
                    continue 
                # pair_status = False
                pair_check = pair.split()
                items_count = len(pair_check)
                  
                if items_count > 1:
                    key_val = pair.rsplit(maxsplit = 1)
                    station_name_codes_master['station_name'].append(key_val[0])
                    station_name_codes_master['station_code'].append(key_val[1])  
            # else:
            #     standby_ls.append(pair)


# print(station_name_codes_master)
# print(standby_ls)
stn_code_page1_df = pd.DataFrame(station_name_codes_master)
print('Dataframed...')

stn_code_page1_df.to_csv(r"D:\01_Research_and_Development\0_Python_Grind\Datasets\station_name_code\page_four.csv" ,index= False)
print("CSV Created!")

['PURNEA PRNA', None, 'SAHIBZADA AJIT SINGH NAGAR SASN\n(MOHALI)', None, 'SINGRAULI SGRL', None, 'TIRUPPUR TUP', None]
['PURULIA PRR', None, None, '', 'SINDHANUR SNNR', None, 'TIRUR TIR', None]
['PURUNAKATAK PUKK', None, 'SAI NAGAR SHIRDI', 'SNSI', 'SIRPUR KAGAZNAGAR SKZR', None, 'TIRUVANNAMALAI TNM', None]
['RAE BARELI RBL', None, 'SALEM SA', None, 'SISWA BAZAR SBZ', None, 'THRISUR TCR', None]
['RAIGARH RIG', None, 'SAMALKOT SLO', None, 'SITAPUR CITY SPC', None, 'THIRUVANTHAPURAM NORTH TVCN', None]
['RAJAHMUNDRY RJY', None, 'SAMASTIPUR SPJ', None, 'SIURI SURI', None, 'TUMSAR ROAD TMR', None]
['RAJA-KI-MANDI RKM', None, 'SAMBALPUR CITY SBPY', None, 'SOGARIYA SGAC', None, 'TUNI TUNI', None]
['RAJGIR RGD', None, 'SAMUKTALA ROAD AMTA', None, 'SOLAN SOL', None, 'UDAIPUR CITY UDZ', None]
['RAJNANDGAON RJN', None, 'SANTRAGACHI SRC', None, 'SOMNATH SMNH', None, 'UDUPI UD', None]
['RAMAGUNDAM RDM', None, 'SARNATH SRNT', None, 'SONPUR SEE', None, 'UNA HIMACHAL UHL', None]
['RAMESWARAM RMM', Non